# Statistical Analysis

> **Purpose**: Demonstrate the `statistics.py` module — compute descriptive stats, detect outliers, and build a correlation matrix.

**All business logic lives in `../statistics.py`. This notebook only imports and calls those functions.**

### Analysis performed
| Function | What it produces |
|----------|------------------|
| `compute_descriptive_stats` | mean, median, std, min, max, skewness, kurtosis per numeric column |
| `compute_numeric_summary` | Z-score outlier count/%, IQR outlier count/%, fence values |
| `compute_categorical_summary` | cardinality, null rate, top value counts per categorical column |
| `compute_missing_summary` | null count + percentage per column |
| `compute_correlation_matrix` | Pearson correlation matrix for numeric columns |

> **Previous step**: `rules.ipynb` | **Next step**: `anomaly.ipynb`

In [ ]:
import sys
import os
import json

sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from cleaning import load_dataset, run_cleaning
from statistics import (
    compute_descriptive_stats,
    compute_numeric_summary,
    compute_categorical_summary,
    compute_missing_summary,
    compute_correlation_matrix,
    run_statistics,
)

DATA_PATH = os.path.join("..", "data", "dataset_ecommerce_transactions_data.csv")
df_raw   = load_dataset(DATA_PATH)
df_clean = run_cleaning(df_raw)
print(f"Dataset loaded and cleaned: {df_clean.shape}")

## 1. Descriptive Statistics

In [ ]:
desc = compute_descriptive_stats(df_clean)
pd.DataFrame(desc).T

## 2. Outlier Detection — Z-Score & IQR

In [ ]:
num_summary = compute_numeric_summary(df_clean)
pd.DataFrame(num_summary).T

## 3. Categorical Column Summary

In [ ]:
cat_summary = compute_categorical_summary(df_clean)
for col, info in cat_summary.items():
    print(f"\n{col}:")
    print(f"  Cardinality : {info['cardinality']}")
    print(f"  Null rate   : {info['null_rate']}%")
    print(f"  Top values  : {list(info['top_values'].keys())[:5]}")

## 4. Missing Value Summary

In [ ]:
missing_summary = compute_missing_summary(df_clean)
missing_df = pd.DataFrame(missing_summary).T
missing_df[missing_df["null_count"] > 0]

## 5. Correlation Matrix

In [ ]:
corr = compute_correlation_matrix(df_clean)
if corr["matrix"]:
    corr_df = pd.DataFrame(corr["matrix"])
    print(corr_df.round(4))
else:
    print("Not enough numeric columns for correlation matrix.")

## Full Statistics Run

In [ ]:
stats_report = run_statistics(df_clean)
print("Keys in statistics report:", list(stats_report.keys()))
print("\n=== Descriptive Stats ===")
print(json.dumps(stats_report["descriptive_stats"], indent=2))

---
## Key Takeaways

- Skewness and kurtosis reveal distribution shape — high skewness suggests outliers or asymmetric data
- Z-score threshold = 3.0 — values beyond ±3 standard deviations are flagged
- IQR fence = Q1 − 1.5×IQR to Q3 + 1.5×IQR (Tukey fences) — a more robust outlier method
- Both outlier methods are applied; the more conservative count should guide cleaning decisions
- Categorical cardinality > expected suggests unknown values (confirm with `rules.ipynb`)
- The correlation matrix helps spot multicollinearity before feature engineering